In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# ----------------------------
# Hyperparameters
# ----------------------------
batch_size = 4          # number of images
large_patches_per_img = 4
M = batch_size * large_patches_per_img  # total large patches
input_dim = 128
embed_dim = 64
tau = 0.07

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# Simple Large Patch Model
# ----------------------------
class SimpleLargePatchModel(nn.Module):
    def __init__(self, input_dim, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, embed_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=1)

model = SimpleLargePatchModel(input_dim, embed_dim).to(device)

# ----------------------------
# Dummy Inputs
# ----------------------------
# Fake large patch features (input to model)
x = torch.randn(M, input_dim).to(device)

# Dummy compositional targets (pretend ground truth from small patches)
# These should be normalized since we use cosine similarity
t = F.normalize(torch.randn(M, embed_dim), dim=1).to(device)

# Dummy augmented view for consistency
x_aug = x + 0.1 * torch.randn_like(x)
x_aug = x_aug.to(device)

# ----------------------------
# Forward Pass
# ----------------------------
f = model(x)            # large patch embeddings
f_aug = model(x_aug)    # augmented embeddings

# ----------------------------
# 1️⃣ Contrastive Loss
# ----------------------------
# Similarity matrix between all f_i and all t_k
sim_matrix = torch.matmul(f, t.T)  # shape: [M, M]

# Scale by temperature
sim_matrix = sim_matrix / tau

# Labels: correct target is diagonal
labels = torch.arange(M).to(device)

# Cross-entropy over rows
L_contrast = F.cross_entropy(sim_matrix, labels)

# ----------------------------
# 2️⃣ Smoothness Loss
# ----------------------------
# Define adjacency pairs (simple grid assumption)
# Here we just connect consecutive patches for demo
L_smooth = 0.0
count = 0

for i in range(M - 1):
    L_smooth += torch.norm(f[i] - f[i + 1], p=2) ** 2
    count += 1

L_smooth = L_smooth / count

# ----------------------------
# 3️⃣ Consistency Loss
# ----------------------------
L_consist = 1 - F.cosine_similarity(f, f_aug, dim=1).mean()

# ----------------------------
# Total Loss
# ----------------------------
lambda_contrast = 1.0
lambda_smooth = 0.1
lambda_consist = 0.5

L_total = (
    lambda_contrast * L_contrast
    + lambda_smooth * L_smooth
    + lambda_consist * L_consist
)

# ----------------------------
# Backprop
# ----------------------------
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

optimizer.zero_grad()
L_total.backward()
optimizer.step()


print("Contrastive Loss:", L_contrast.item())
print("Smoothness Loss:", L_smooth.item())
print("Consistency Loss:", L_consist.item())
print("Total Loss:", L_total.item())